In [1]:
import torch
import torch.nn as nn 
import math
import random
import numpy as np
import cv2

In [2]:
data = np.load(f"lego_200x200.npz")

# Training images: [100, 200, 200, 3]
images_train = data["images_train"] / 255.0

# Cameras for the training images 
# (camera-to-world transformation matrix): [100, 4, 4]
c2ws_train = data["c2ws_train"]

# Validation images: 
images_val = data["images_val"] / 255.0

# Cameras for the validation images: [10, 4, 4]
# (camera-to-world transformation matrix): [10, 200, 200, 3]
c2ws_val = data["c2ws_val"]

# Test cameras for novel-view video rendering: 
# (camera-to-world transformation matrix): [60, 4, 4]
c2ws_test = data["c2ws_test"]

# Camera focal length
focal = data["focal"] 

In [27]:
def transform(c2w, x_c):
    #return (c2w @ x_c.T).T
    R = c2w[:3, :3] 
    t = c2w[:3, 3]  
    x_w = (R @ x_c.T).T + t
    return x_w

def verify(x, c2w): 
    return x == transform(np.linalg.inv(c2w), transform(c2w, x))

In [28]:
c2w = np.array([
    [0, -1, 0, 1],
    [1,  0, 0, 2],
    [0,  0, 1, 3],
    [0,  0, 0, 1]
])
x_c = np.array([1, 0, 0]) 
print("Verification passed:", verify(x_c, c2w))


Verification passed: [ True  True  True]


In [29]:
def pixel_to_camera(K, uv, s):
    uv = np.atleast_2d(uv)  
    ones = np.ones((uv.shape[0], 1))
    uv_h = np.hstack([uv, ones]) 
    s_uv = s * uv_h
    return (np.linalg.inv(K) @ s_uv.T).T
    

In [30]:
def pixel_to_ray(K, c2w, uv): 
    x_c = pixel_to_camera(K, uv, 1.0)
    x_w = transform(c2w, x_c)
    r_o = c2w[:3, 3]
    r_d = x_w - r_o
    r_d_normalized = r_d / np.linalg.norm(r_d, axis=-1, keepdims=True)
    return r_o, r_d_normalized

In [31]:
class RaysData: 
    def __init__(self, images, K, c2ws):
        self.images = images
        self.K = K
        self.c2ws = c2ws
        self.H, self.W = images[0].shape[:2]  # Assume all images same size
        self.total_imgs = len(images)

        x_coords = np.arange(self.W)
        y_coords = np.arange(self.H)
        xx, yy = np.meshgrid(x_coords, y_coords, indexing='xy') 
        self.uvs = np.column_stack([xx.ravel(), yy.ravel()]) 
        self.pixels = images[0][self.uvs[:, 1], self.uvs[:, 0]] 
        self.rays_o = []
        self.rays_d = [] 
        for idx in range(self.total_imgs):
            c2w = self.c2ws[idx]
            
            uv_pixels = self.uvs + 0.5  
            r_o, r_d = pixel_to_ray(self.K, c2w, uv_pixels)
            
            self.rays_o.extend([r_o] * len(self.uvs))  
            self.rays_d.extend(r_d)  
        
        self.rays_o = np.array(self.rays_o)  
        self.rays_d = np.array(self.rays_d) 

    def sample_rays(self, N, M=3):
        img_idx = np.random.choice(self.total_imgs, size=M, replace=False)
        num_ray = N // M

        batch_colors = []
        batch_ray_o = []
        batch_ray_d = []
        
        for idx in img_idx: 
            image = self.images[idx]
            c2w = self.c2ws[idx]
            total_pixels = self.H * self.W

            batch_indices = np.random.choice(total_pixels, size=num_ray, replace=False)
            batch_y = batch_indices // self.W 
            batch_x = batch_indices % self.W 
            uv = np.column_stack([batch_x + 0.5, batch_y + 0.5])
            r_o, r_d = pixel_to_ray(self.K, c2w, uv) 
            batch_ray_o.extend([r_o] * num_ray)
            batch_ray_d.extend(r_d)
            batch_colors.extend(image[batch_y, batch_x])

        pixels = np.array(batch_colors) / 255.0
        rays_o = np.array(batch_ray_o)
        rays_d = np.array(batch_ray_d)
        return rays_o, rays_d, pixels

In [ ]:
def sample_along_rays(ray_o, ray_d, near=2.0, far=6.0, n_samples=32, random=True):
    # Convert inputs to numpy arrays if they're tensors
    if isinstance(ray_o, torch.Tensor):
        ray_o = ray_o.cpu().numpy()
    if isinstance(ray_d, torch.Tensor):
        ray_d = ray_d.cpu().numpy()
    
    N = ray_o.shape[0] 
    t = np.linspace(near, far, n_samples)
    t_width = (far - near) / (n_samples - 1)
    if random: 
        t = t + np.random.rand(n_samples) * t_width
    
    points_3d = np.zeros((N, n_samples, 3))

    for ray_idx in range(N):
        for sample_idx in range(n_samples):
            points_3d[ray_idx, sample_idx] = (
                ray_o[ray_idx] + ray_d[ray_idx] * t[sample_idx]
            )   
    
    return points_3d

In [ ]:
def sample_along_rays_2(ray_o, ray_d, near=2.0, far=6.0, n_samples=32, random=True):
    device = ray_o.device
    N = ray_o.shape[0]

    t_vals = torch.linspace(near, far, n_samples, device=device)
    if random:
        mids = 0.5 * (t_vals[:-1] + t_vals[1:])
        t_rand = torch.rand((N, n_samples), device=device)
        t_vals = mids.unsqueeze(0) + (far - near) / n_samples * t_rand
    else:
        t_vals = t_vals.expand(N, n_samples)

    points = ray_o.unsqueeze(1) + ray_d.unsqueeze(1) * t_vals.unsqueeze(-1)
    return points  # shape: [N, n_samples, 3]

In [105]:
import viser, time  # pip install viser
import numpy as np

# --- You Need to Implement These ------
H, W = images_train.shape[1:3] 
K=np.array([[focal, 0, W/2],    
    [0, focal, H/2],    
    [0, 0, 1]
])

dataset = RaysData(images_train, K, c2ws_train)
rays_o, rays_d, pixels = dataset.sample_rays(100, M=30) # Should expect (B, 3)
points = sample_along_rays(rays_o, rays_d)
H, W = images_train.shape[1:3]
# ---------------------------------------

server = viser.ViserServer(share=True)
for i, (image, c2w) in enumerate(zip(images_train, c2ws_train)):
    server.add_camera_frustum(
        f"/cameras/{i}",
        fov=2 * np.arctan2(H / 2, K[0, 0]),
        aspect=W / H,
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=image
    )
for i, (o, d) in enumerate(zip(rays_o, rays_d)):
    server.add_spline_catmull_rom(
        f"/rays/{i}", positions=np.stack((o, o + d * 6.0)),
    )


server.add_point_cloud(
    f"/samples",
    colors=np.zeros_like(points).reshape(-1,3),
    points=points.reshape(-1, 3),
    point_size=0.02,
)

while True:
    time.sleep(0.1)  # Wait to allow visualization to run

AttributeError: 'numpy.ndarray' object has no attribute 'unsqueeze'

In [ ]:
# Visualize Cameras, Rays and Samples
import viser, time
import numpy as np

# --- You Need to Implement These ------
dataset = RaysData(images_train, K, c2ws_train)

# This will check that your uvs aren't flipped
uvs_start = 0
uvs_end = 40_000
sample_uvs = dataset.uvs[uvs_start:uvs_end] # These are integer coordinates of widths / heights (xy not yx) of all the pixels in an image
# uvs are array of xy coordinates, so we need to index into the 0th image tensor with [0, height, width], so we need to index with uv[:,1] and then uv[:,0]
assert np.all(images_train[0, sample_uvs[:,1], sample_uvs[:,0]] == dataset.pixels[uvs_start:uvs_end])

# # Uncoment this to display random rays from the first image
# indices = np.random.randint(low=0, high=40_000, size=100)

# Uncomment this to display random rays from the top left corner of the image
indices_x = np.random.randint(low=100, high=200, size=100)
indices_y = np.random.randint(low=0, high=100, size=100)
indices = indices_x + (indices_y * 200)

data = {"rays_o": dataset.rays_o[indices], "rays_d": dataset.rays_d[indices]}
points = sample_along_rays(data["rays_o"], data["rays_d"], random=True)
# ---------------------------------------

server = viser.ViserServer(share=True)
for i, (image, c2w) in enumerate(zip(images_train, c2ws_train)):
  server.add_camera_frustum(
    f"/cameras/{i}",
    fov=2 * np.arctan2(H / 2, K[0, 0]),
    aspect=W / H,
    scale=0.15,
    wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
    position=c2w[:3, 3],
    image=image
  )
for i, (o, d) in enumerate(zip(data["rays_o"], data["rays_d"])):
  positions = np.stack((o, o + d * 6.0))
  server.add_spline_catmull_rom(
      f"/rays/{i}", positions=positions,
  )
server.add_point_cloud(
    f"/samples",
    colors=np.zeros_like(points).reshape(-1, 3),
    points=points.reshape(-1, 3),
    point_size=0.03,
)

while True:
    time.sleep(0.1)  # Wait to allow visualization to run

╭────── viser (listening *:8082) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8082   │
│   Websocket │ ws://localhost:8082     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://midtone-card.share.viser.studio

C:\Users\isabe\AppData\Local\Temp\ipykernel_20512\3225039921.py:29: DeprecationWarning: ViserServer.add_camera_frustum has been deprecated, use ViserServer.scene.add_camera_frustum instead. Alternatively, pin to `viser<0.2.0`.
  server.add_camera_frustum(
C:\Users\isabe\AppData\Local\Temp\ipykernel_20512\3225039921.py:40: DeprecationWarning: ViserServer.add_spline_catmull_rom has been deprecated, use ViserServer.scene.add_spline_catmull_rom instead. Alternatively, pin to `viser<0.2.0`.
  server.add_spline_catmull_rom(
C:\Users\isabe\AppData\Local\Temp\ipykernel_20512\3225039921.py:43: DeprecationWarning: ViserServer.add_point_cloud has been deprecated, use ViserServer.scene.add_point_cloud instead. Alternatively, pin to `viser<0.2.0`.
  server.add_point_cloud(


AttributeError: 'Tensor' object has no attribute 'astype'

(viser) Connection opened (0, 1 total), 606 persistent messages

(viser) Connection closed (0, 0 total)

(viser) Connection opened (1, 1 total), 606 persistent messages

(viser) Connection closed (1, 0 total)

In [75]:
def PE(x, L=10): 
    #single coordinate 
    encoded = [x]

    for i in range(L):
        freq = 2**i * torch.pi
        encoded.append(torch.sin(freq * x))
        encoded.append(torch.cos(freq * x))

    return torch.cat(encoded, dim=-1)


def positional_encoding_3d(coords, L = 10): 
    if isinstance(coords, np.ndarray):
        coords = torch.from_numpy(coords).float()

    x, y, z= coords[..., 0:1], coords[..., 1:2], coords[..., 2:3]
    x_encoded = PE(x, L)
    y_encoded = PE(y, L)
    z_encoded = PE(z, L)
    return torch.cat([x_encoded, y_encoded, z_encoded], dim=-1)

In [106]:
class NeRFModel(nn.Module): 
    def __init__(self, width=256, pos_L=10, dir_L=4):
        super().__init__()

        self.pos_enc_dim = 3*(2*pos_L + 1)
        self.dir_enc_dim = 3*(2*dir_L + 1)

        self.layer1 = nn.Linear(self.pos_enc_dim, width)
        self.layer2 = nn.Linear(width, width)
        self.layer3 = nn.Linear(width, width)
        self.layer4 = nn.Linear(width, width)

        self.layer5 = nn.Linear(width + self.pos_enc_dim, width)
        self.layer6 = nn.Linear(width, width)
        self.layer7 = nn.Linear(width, width)
        self.layer8 = nn.Linear(width, width)

        self.density_out = nn.Linear(width, 1)

        self.color_layer1 = nn.Linear(width, width)
        self.color_layer2 = nn.Linear(width + self.dir_enc_dim, width//2)
        self.color_out = nn.Linear(width//2, 3)

        #Activation
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, d): 
        x_encoded = positional_encoding_3d(x, L=10)
        d_encoded = positional_encoding_3d(d, L=4)

        h = self.relu(self.layer1(x_encoded))
        h = self.relu(self.layer2(h))
        h = self.relu(self.layer3(h))
        h = self.relu(self.layer4(h))
        h = torch.cat([h, x_encoded], dim=-1)
        h = self.relu(self.layer5(h))
        h = self.relu(self.layer6(h))
        h = self.relu(self.layer7(h))
        h8 = self.layer8(h)
        density = self.relu(self.density_out(h8))

        h_color = self.color_layer1(h8)
        h_color = torch.cat([h_color, d_encoded], dim=-1)
        h_color = self.relu(self.color_layer2(h_color))
        rgb = self.sigmoid(self.color_out(h_color))

        return rgb, density


In [107]:
def volrend(sigmas, rgbs, step_size):
    deltas = torch.ones_like(sigmas) * step_size
    cumulative_sigma_delta = torch.cumsum(sigmas * deltas, dim=1)
    T = torch.exp(-(cumulative_sigma_delta - sigmas * deltas))
    alphas = 1-torch.exp(-sigmas * deltas)
    weights = T * alphas
    rendered_colors = torch.sum(weights * rgbs, dim =1)

    return rendered_colors
def compute_psnr(mse_loss):
    return 20 * torch.log10(1.0 / torch.sqrt(mse_loss))

In [ ]:

import matplotlib.pyplot as plt

def render_validation_view(model, iteration, camera_idx=0, device=torch.device('cpu'), show='val'):
    if show == 'val': 
      c2w = c2ws_val
      image = images_val
    elif show == 'train':
      c2w = c2ws_train
      image = images_train
    
    model.eval()

    with torch.no_grad():
        c2w = c2w[camera_idx]
        H, W = image.shape[1:3]

        rays_o, rays_d = create_rays_for_camera(K, c2w, H, W, device=device)

        points_3d = sample_along_rays(rays_o, rays_d, n_samples=64, random=False)
        points_flat = points_3d.reshape(-1,3)
        rays_d_repeated = rays_d.unsqueeze(1).repeat(1, 64, 1).reshape(-1, 3)


        rgbs, densities = model(points_flat, rays_d_repeated)
        rgbs_reshaped = rgbs.reshape(H*W, 64, 3)
        densities_reshaped = densities.reshape(H*W, 64, 1)

        step_size= (0.55-0.02)/64
        rendered_image = volrend(densities_reshaped, rgbs_reshaped, step_size)
        rendered_image = rendered_image.reshape(H, W, 3).cpu().numpy()
        rendered_image = np.clip(rendered_image, 0, 1)

        plt.figure(figsize=(2, 2))  # Optional: adjust size as needed
        plt.imshow(rendered_image)
        #plt.title("Neural Field Output")
        plt.axis('off')
        plt.tight_layout()
        plt.show()


    model.train()
    return rendered_image

In [109]:
def create_rays_for_camera(K, c2w, H, W):  #device=torch.device('cpu')
    u = np.arange(W) + 0.5
    v = np.arange(H) + 0.5
    uu, vv = np.meshgrid(u, v) 
    uv = np.column_stack([uu.ravel(), vv.ravel()]) 
    rays_o, rays_d = pixel_to_ray(K, c2w, uv)

    rays_o_tensor = torch.FloatTensor(rays_o) #.to(device)
    rays_d_tensor = torch.FloatTensor(rays_d) #.to(device)
#     if rays_o_tensor.dim() == 1:
#             rays_o_tensor = rays_o_tensor.reshape(-1, 3)
#     if rays_d_tensor.dim() == 1:
#             rays_d_tensor = rays_d_tensor.reshape(-1, 3)
        
    #print(f"create_rays_for_camera: rays_o shape {rays_o_tensor.shape}, rays_d shape {rays_d_tensor.shape}")
    return rays_o_tensor, rays_d_tensor

In [110]:
import matplotlib.pyplot as plt 

def render_validation_view(model, iteration, camera_idx=0): #device=torch.device('cpu')
    model.eval()

    with torch.no_grad(): 
        c2w = c2ws_val[camera_idx]
        H, W = images_val.shape[1:3]

        rays_o, rays_d = create_rays_for_camera(K, c2w, H, W) #device=device

        print(f"Validation - rays_o device: {rays_o.device}, rays_d device: {rays_d.device}")

        print("rays_d shape:", rays_d.shape)
        points_3d = sample_along_rays(rays_o, rays_d, n_samples=32, random=False)
        print("points_3d shape:", points_3d.shape)
        points_flat = points_3d.reshape(-1,3)
        rays_d_repeated = rays_d.unsqueeze(1).repeat(1, 32, 1).reshape(-1, 3) 
        print("points_flat shape:", points_flat.shape)
        print("rays_d_repeated shape:", rays_d_repeated.shape)


        print(f"Validation - points_flat device: {points_flat.device}")
        print(f"Validation - rays_d_repeated device: {rays_d_repeated.device}")
        print(f"Validation - model device: {next(model.parameters()).device}")

        rgbs, densities = model(points_flat, rays_d_repeated)
        rgbs_reshaped = rgbs.reshape(H*W, 32, 3)
        densities_reshaped = densities.reshape(H*W, 32, 1)

        step_size= (6.0-2.0)/32
        rendered_image = volrend(densities_reshaped, rgbs_reshaped, step_size)
        rendered_image = rendered_image.reshape(H, W, 3).cpu().numpy()
        rendered_image = np.clip(rendered_image, 0, 1)
        
        plt.figure(figsize=(6, 6))  # Optional: adjust size as needed
        plt.imshow(rendered_image)
        #plt.title("Neural Field Output")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    
    model.train() 

In [111]:
import time



def train_nerf_3d(width=256, learning_rate=5e-4, track_psnr=True): 
    #device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #print(f"Using device: {device}")
    dataset = RaysData(images_train, K, c2ws_train)
    
    model = NeRFModel(width=width, pos_L=10, dir_L=4) #.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()

    psnr_values = []
    iteration_numbers = []

    for iteration in range(3000):
        start = time.time()
        rays_o, rays_d, pixels = dataset.sample_rays(10000, M=10)
        print("Sampling time:", time.time() - start)

        
        rays_o = torch.FloatTensor(rays_o)#.to(device)
        rays_d = torch.FloatTensor(rays_d)#.to(device)
        pixels = torch.FloatTensor(pixels)#.to(device)

        start = time.time()
        points_3d = sample_along_rays(rays_o, rays_d, n_samples=32, random=True)
        print("Sampling along rays:", time.time() - start)



        # if points_3d.device != device:
        #     points_3d = points_3d.to(device)
        points_flatten = points_3d.reshape(-1,3)

        rays_d_repeated = rays_d.unsqueeze(1).repeat(1, 32, 1).reshape(-1, 3) 

        start = time.time()
        rgbs, densities = model(points_flatten, rays_d_repeated)
        print("Model forward pass:", time.time() - start)


        rgbs_reshaped = rgbs.reshape(10000,32,3)
        densities_reshaped = densities.reshape(10000,32,1)
       
        step_size = (6.0-2.0)/32

        start = time.time()
        rendered_color = volrend(densities_reshaped, rgbs_reshaped, step_size)
        print("Volume rendering:", time.time() - start)

        loss = loss_fn(rendered_color, pixels)

        optimizer.zero_grad()

        loss.backward()  
        optimizer.step() 
        print(f"iteration {iteration} complete")
        if iteration % 10 == 0 and track_psnr:
                psnr = compute_psnr(loss)
                psnr_values.append(psnr.item())
                iteration_numbers.append(iteration)

        if iteration % 100 == 0:
            psnr = compute_psnr(loss)
            print(f"Iteration {iteration}, Loss: {loss.item():.6f}, PSNR: {psnr.item():.2f} dB")

            if iteration % 500 == 0:
                print("Generating visualization...")
                render_validation_view(model,iteration) #, device=device)
    print("Final result visualization...")
    render_validation_view(model,iteration)#, device=device)
    return  model, psnr_values, iteration_numbers

In [112]:
model, psnr_values, iterations = train_nerf_3d()

Sampling time: 0.024264097213745117
Sampling along rays: 0.005985260009765625


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 310000 but got size 320000 for tensor number 1 in the list.

In [ ]:

import time
import numpy as np
import torch
import matplotlib.pyplot as plt

def srgb_to_linear_np(x):
    # x: numpy array in [0,1]
    a = 0.055
    mask = x <= 0.04045
    linear = np.where(mask, x / 12.92, ((x + a) / (1 + a)) ** 2.4)
    return linear
def train_nerf_3d_val(
    width=256,
    learning_rate=5e-4,
    track_psnr=True,
    total_iters=1500,
    eval_every=50,
    rays_per_step=10000,
    M=10,
    n_samples=64,
    device=None
):
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    dataset = RaysData(images_train, K, c2ws_train)  # uses global/outer-scope images_train, K, c2ws_train

    model = NeRFModel(width=width, pos_L=10, dir_L=4).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.MSELoss()

    psnr_list = []       # will contain lists/arrays of length 10 (per checkpoint)
    iter_numbers = []

    for iteration in range(total_iters):
        start = time.time()
        rays_o, rays_d, pixels = dataset.sample_rays(rays_per_step, M=M)  # existing API
        assert rays_o.shape == (rays_per_step, 3)
        assert rays_d.shape == (rays_per_step, 3)
        
        # convert to torch
        rays_o_t = torch.FloatTensor(rays_o).to(device)
        rays_d_t = torch.FloatTensor(rays_d).to(device)
        pixels_t = torch.FloatTensor(pixels).to(device)

        # sample points along rays (torch implementation; your function)
        points_3d = sample_along_rays(rays_o_t, rays_d_t, n_samples=n_samples, random=True)
        assert points_3d.shape == (rays_per_step, n_samples, 3)
        if points_3d.device != device:
            points_3d = points_3d.to(device)
        points_flatten = points_3d.reshape(-1, 3)
        rays_d_repeated = rays_d_t.unsqueeze(1).repeat(1, n_samples, 1).reshape(-1, 3)

        # model forward
        rgbs, densities = model(points_flatten, rays_d_repeated)
        # reshape back
        B = rays_per_step
        rgbs_reshaped = rgbs.reshape(B, n_samples, 3)
        densities_reshaped = densities.reshape(B, n_samples, 1)

        step_size = (6.0 - 2.0) / float(n_samples)

        rendered_color = volrend(densities_reshaped, rgbs_reshaped, step_size)  # expects (B,3)

        loss = loss_fn(rendered_color, pixels_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if iteration % 10 == 0:
            # quick status print
            try:
                # compute PSNR from loss on this training batch (optional)
                
                psnr = compute_psnr(loss)
                print(f"Iter {iteration:04d} | train loss {loss.item():.6f} | train PSNR {psnr.item():.2f} dB")
            except Exception:
                print(f"Iter {iteration:04d} | train loss {loss.item():.6e}")


        
        # Evaluate on validation set every eval_every iterations (and at end)
        if track_psnr and (iteration % eval_every == 0 or iteration == total_iters - 1):
            model.eval()
            per_image_psnrs = []
            # Render each validation image deterministically (random=False)
            for vidx in range(len(c2ws_val)):  # expected 10
                rendered = render_validation_view(model, iteration, camera_idx=vidx, device=device)
                print("Rendered range:", rendered.min(), "to", rendered.max())
                gt = images_val[vidx]  # numpy in [0,1]
                gt = srgb_to_linear_np(gt)
                print("GT range:", gt.min(), "to", gt.max())

                mse = np.mean((rendered.astype(np.float32) - gt.astype(np.float32)) ** 2)
                mse_tensor = torch.tensor(mse, dtype=torch.float32)

                psnr = compute_psnr(mse_tensor)
                per_image_psnrs.append(psnr)
            per_image_psnrs = np.array(per_image_psnrs, dtype=np.float32)  # (10,)
            psnr_list.append(per_image_psnrs)
            iter_numbers.append(iteration)

            mean_psnr = float(np.mean(per_image_psnrs))
            print(f"[VAL] Iter {iteration:04d} | per-image PSNRs: {np.round(per_image_psnrs, 3).tolist()} | mean PSNR: {mean_psnr:.3f} dB")
            model.train()

    # convert list to numpy array shape (n_checkpoints, 10)
    if len(psnr_list) > 0:
        psnr_array = np.stack(psnr_list, axis=0)
    else:
        psnr_array = np.zeros((0, len(c2ws_val)), dtype=np.float32)

    return model, psnr_array, iter_numbers


NameError: name 'model' is not defined

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

def srgb_to_linear_np(x):
    # x: numpy array in [0,1]
    a = 0.055
    mask = x <= 0.04045
    linear = np.where(mask, x / 12.92, ((x + a) / (1 + a)) ** 2.4)
    return linear
def train_nerf_3d_val(
    width=256,
    learning_rate=5e-4,
    track_psnr=True,
    total_iters=1500,
    eval_every=50,
    rays_per_step=10000,
    M=10,
    n_samples=64,
    device=None
):
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    dataset = RaysData(images_train, K, c2ws_train)  # uses global/outer-scope images_train, K, c2ws_train

    model = NeRFModel(width=width, pos_L=10, dir_L=4).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.MSELoss()

    psnr_list = []       # will contain lists/arrays of length 10 (per checkpoint)
    iter_numbers = []

    for iteration in range(total_iters):
        start = time.time()
        rays_o, rays_d, pixels = dataset.sample_rays(rays_per_step, M=M)  # existing API
        assert rays_o.shape == (rays_per_step, 3)
        assert rays_d.shape == (rays_per_step, 3)
        
        # convert to torch
        rays_o_t = torch.FloatTensor(rays_o).to(device)
        rays_d_t = torch.FloatTensor(rays_d).to(device)
        pixels_t = torch.FloatTensor(pixels).to(device)

        # sample points along rays (torch implementation; your function)
        points_3d = sample_along_rays(rays_o_t, rays_d_t, n_samples=n_samples, random=True)
        assert points_3d.shape == (rays_per_step, n_samples, 3)
        if points_3d.device != device:
            points_3d = points_3d.to(device)
        points_flatten = points_3d.reshape(-1, 3)
        rays_d_repeated = rays_d_t.unsqueeze(1).repeat(1, n_samples, 1).reshape(-1, 3)

        # model forward
        rgbs, densities = model(points_flatten, rays_d_repeated)
        # reshape back
        B = rays_per_step
        rgbs_reshaped = rgbs.reshape(B, n_samples, 3)
        densities_reshaped = densities.reshape(B, n_samples, 1)

        step_size = (6.0 - 2.0) / float(n_samples)

        rendered_color = volrend(densities_reshaped, rgbs_reshaped, step_size)  # expects (B,3)

        loss = loss_fn(rendered_color, pixels_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if iteration % 10 == 0:
            # quick status print
            try:
                # compute PSNR from loss on this training batch (optional)
                
                psnr = compute_psnr(loss)
                print(f"Iter {iteration:04d} | train loss {loss.item():.6f} | train PSNR {psnr.item():.2f} dB")
            except Exception:
                print(f"Iter {iteration:04d} | train loss {loss.item():.6e}")


        
        # Evaluate on validation set every eval_every iterations (and at end)
        if track_psnr and (iteration % eval_every == 0 or iteration == total_iters - 1):
          model.eval()
          per_image_psnrs = []
          per_image_maes = []  # Track MAE too
          per_image_ssims = []  # Track structural similarity
          
          for vidx in range(len(c2ws_val)):
              rendered = render_validation_view(model, iteration, camera_idx=vidx, device=device)
              rendered = np.clip(rendered, 0.0, 1.0)
              gt = images_val[vidx]
              gt = srgb_to_linear_np(gt)
              
              # Multiple quality metrics
              mse = np.mean((rendered - gt) ** 2)
              mae = np.mean(np.abs(rendered - gt))  # Mean Absolute Error
              psnr = compute_psnr_numpy(mse)
              
              # Try computing SSIM (structural similarity)
              try:
                  from skimage.metrics import structural_similarity as ssim
                  ssim_val = ssim(rendered, gt, channel_axis=2, data_range=1.0)
                  per_image_ssims.append(ssim_val)
              except:
                  ssim_val = 0
                  per_image_ssims.append(0)
              
              per_image_psnrs.append(psnr)
              per_image_maes.append(mae)
          
          # Print all metrics///
          avg_psnr = float(np.mean(per_image_psnrs))
          avg_mae = float(np.mean(per_image_maes))
          avg_ssim = float(np.mean(per_image_ssims))
          
          print(f"[VAL] Iter {iteration:04d} | "
                f"PSNR: {avg_psnr:.2f} dB | "
                f"MAE: {avg_mae:.4f} | "
                f"SSIM: {avg_ssim:.4f}")
    
    # If PSNR decreases but MAE improves, it's likely brightness/contrast issues
          if len(psnr_list) > 0:
              prev_psnr = np.mean(psnr_list[-1])
              if avg_psnr < prev_psnr and avg_mae < prev_mae:
                  print("⚠️  PSNR decreased but MAE improved - likely global brightness/contrast shift")
          print(f"[VAL] Iter {iteration:04d} | per-image PSNRs: {np.round(per_image_psnrs, 3).tolist()} | mean PSNR: {mean_psnr:.3f} dB")
          model.train()

    # convert list to numpy array shape (n_checkpoints, 10)
    if len(psnr_list) > 0:
        psnr_array = np.stack(psnr_list, axis=0)
    else:
        psnr_array = np.zeros((0, len(c2ws_val)), dtype=np.float32)

    return model, psnr_array, iter_numbers
